# 🎵 《儲互社互助島》日夜雙原創配樂 Colab GPU 產線
請確認上方選單：**執行階段 → 變更執行階段類型 → T4 GPU**，接著按「全部執行」即可一鍵產出並自動下載：
1. `bgm_day.mp3` — 白天暖心微縮城鎮風（輕快烏克麗麗＋馬林巴木琴＋清脆空心吉他，動物森友會感無縫循環）
2. `bgm_night.mp3` — 🌌 星空夜景晚安曲（溫柔星空八音盒＋療癒輕鋼琴無縫循環）

下載後放進專案資料夾的 `assets/audio/`，遊戲內切日夜景即自動淡入淡出切換！

In [ ]:
!pip -q install transformers scipy pydub accelerate
import os, torch, numpy as np
import scipy.io.wavfile as wav
from transformers import AutoProcessor, MusicgenForConditionalGeneration
from pydub import AudioSegment
from google.colab import files

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

processor = AutoProcessor.from_pretrained('facebook/musicgen-small')
model = MusicgenForConditionalGeneration.from_pretrained('facebook/musicgen-small').to(device)
sr = model.config.audio_encoder.sampling_rate

def make_loopable_mp3(prompt, out_mp3, tokens=820):
    inputs = processor(text=[prompt], padding=True, return_tensors='pt').to(device)
    with torch.no_grad():
        audio_values = model.generate(**inputs, do_sample=True, guidance_scale=3.2, max_new_tokens=tokens)
    arr = audio_values[0, 0].cpu().numpy()
    arr = arr / (np.max(np.abs(arr)) + 1e-6) * 0.92
    wav_path = out_mp3.replace('.mp3', '.wav')
    wav.write(wav_path, sr, (arr * 32767).astype(np.int16))
    seg = AudioSegment.from_wav(wav_path)
    # Smooth crossfade tail into head for seamless web looping
    xfade = 1800
    if len(seg) > xfade * 3:
        body = seg[:-xfade]
        tail = seg[-xfade:]
        looped = body.overlay(tail, position=0)
    else:
        looped = seg.fade_in(500).fade_out(800)
    looped.export(out_mp3, format='mp3', bitrate='160k')
    print('Generated:', out_mp3, len(looped), 'ms')

make_loopable_mp3(
    'cheerful warm Taiwanese miniature town afternoon music, bright ukulele, cozy marimba, gentle acoustic guitar, soft shaker, Animal Crossing cozy village vibe, happy clean instrumental, no vocals',
    'bgm_day.mp3'
)

make_loopable_mp3(
    'peaceful starry night music box and soft felt piano lullaby, gentle warm celesta, calm miniature island night atmosphere, Studio Ghibli cozy evening instrumental, relaxing loopable, no vocals, no drums',
    'bgm_night.mp3'
)

files.download('bgm_day.mp3')
files.download('bgm_night.mp3')
